# GPT-3 복습과제 — In-context Learning 이해하기

**논문:** Brown et al., *Language Models are Few-Shot Learners* (NeurIPS 2020)

이 과제에서는 GPT-3 175B를 재현하지 않습니다. 대신 작은 공개 causal language model을 이용해 **zero-shot / one-shot / few-shot prompt의 차이**를 확인하고, 논문의 주요 결과와 benchmark contamination 문제를 간단히 복습합니다.

| Part | 내용 | 논문 연결 |
|---|---|---|
| 1 | Zero / One / Few-shot prompt 구성 | §2, Figure 2.1 |
| 2 | 작은 LM으로 in-context example의 영향 관찰 | §1–3 |
| 3 | 논문 결과 수치 해석 | §3 |
| 4 | Benchmark contamination 간단 실습 | §4 |
| 5 | 서술형 복습 | §5–8 |

> 목표는 코드를 어렵게 구현하는 것이 아니라, **무엇이 바뀌고 무엇은 학습되지 않는지** 이해하는 것입니다.

In [1]:
# 0. 환경 설정
!pip -q install transformers

import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cpu


---
## Part 1. Zero-shot / One-shot / Few-shot prompt 만들기

GPT-3 논문에서 세 설정의 차이는 **inference 시 제공하는 demonstration의 개수**입니다. 세 경우 모두 downstream task를 위해 model weight를 업데이트하지 않습니다.

아래에서는 간단한 영어→프랑스어 번역 task를 예로 들어 prompt를 만들어 봅니다.

### 문제 1-1. `build_prompt`의 빈칸을 채우세요

- `k=0`: instruction과 query만 사용
- `k=1`: demonstration 1개 사용
- `k>1`: 여러 demonstration 사용

In [2]:
examples = [
    ("sea otter", "loutre de mer"),
    ("peppermint", "menthe poivree"),
    ("plush giraffe", "girafe peluche"),
]

def build_prompt(examples, query, k=0):
    prompt = "Translate English to French.\n"
    for src, tgt in examples[:k]:                  # TODO ① 앞에서 k개의 demonstration 사용
        prompt += f"English: {src}\nFrench: {tgt}\n"
    prompt += f"English: {query}\nFrench:"
    return prompt

for k in [0, 1, 3]:
    print(f"\n--- k={k} ---")
    print(build_prompt(examples, "cheese", k))


--- k=0 ---
Translate English to French.
English: cheese
French:

--- k=1 ---
Translate English to French.
English: sea otter
French: loutre de mer
English: cheese
French:

--- k=3 ---
Translate English to French.
English: sea otter
French: loutre de mer
English: peppermint
French: menthe poivree
English: plush giraffe
French: girafe peluche
English: cheese
French:


In [3]:
# ✅ 확인 셀
assert "sea otter" not in build_prompt(examples, "cheese", 0)
assert "sea otter" in build_prompt(examples, "cheese", 1)
assert "plush giraffe" in build_prompt(examples, "cheese", 3)
print("✅ prompt 구성 확인")

✅ prompt 구성 확인


### 문제 1-2 (서술형)
Zero-shot → One-shot → Few-shot으로 갈 때 **변하는 것**과 **변하지 않는 것**을 각각 쓰세요. 특히 model parameter의 update 여부를 포함해 설명하세요.

**답:**

- **변하는 것:** inference 시점에 prompt(context window) 안에 넣어 주는 **demonstration(예시)의 개수**뿐입니다. zero-shot은 task 설명(instruction)과 query만, one-shot은 예시 1개, few-shot은 context window(GPT-3는 2048 tokens)에 들어가는 만큼 보통 10~100개의 예시를 넣습니다. 그 결과 입력 길이와, 모델이 task 형식(입력→출력 패턴)을 알아낼 수 있는 단서의 양이 달라집니다.
- **변하지 않는 것:** **model parameter(weight)는 전혀 update되지 않습니다.** 세 설정 모두 gradient 계산이나 backpropagation 없이, 같은 pretrained 모델의 forward pass만으로 다음 token을 예측합니다. 모델 구조, 사전학습된 weight, 다음 token을 예측하는 방식(학습 목표)도 모두 그대로입니다. 예시는 weight가 아니라 **conditioning context**로만 작용하므로, 다음 prompt로 넘어가면 그 예시의 영향은 남지 않습니다.

---
## Part 2. 작은 Language Model로 In-context Learning 관찰하기

실제 GPT-3는 공개 weight가 없고 175B 모델을 수업 과제로 재현하기도 어렵습니다. 여기서는 가벼운 `distilgpt2`를 사용해 **prompt에 example을 추가했을 때 출력이 어떻게 달라지는지**만 관찰합니다.

> 작은 모델의 결과가 GPT-3 논문의 성능을 재현하는 것은 아닙니다. 출력이 좋지 않아도 정상입니다.

In [4]:
MODEL_ID = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(device).eval()
tokenizer.pad_token = tokenizer.eos_token

@torch.no_grad()
def generate(prompt, max_new_tokens=12):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

print("model loaded:", MODEL_ID)

W1116 20:02:14.161000 75704 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

model loaded: distilgpt2


### 문제 2-1. 같은 query를 zero / one / few-shot으로 비교하세요
`k` 값만 바꾸고 model weight는 전혀 학습하지 않습니다.

In [5]:
query = "cheese"

for k in [0, 1, 3]:
    prompt = build_prompt(examples, query, k)                          # TODO ② build_prompt로 k-shot prompt 생성
    result = generate(prompt)
    print(f"\n[{k}-shot]")
    print(result)


[0-shot]
Translate English to French.
English: cheese
French: cheese
French: cheese
French: cheese
French:

[1-shot]
Translate English to French.
English: sea otter
French: loutre de mer
English: cheese
French: loutre de mer
English: cheese
French:



[3-shot]
Translate English to French.
English: sea otter
French: loutre de mer
English: peppermint
French: menthe poivree
English: plush giraffe
French: girafe peluche
English: cheese
French: coterie
French: coterie
French:


### 문제 2-2 (서술형)
위 실험에서 few-shot 결과가 zero-shot보다 좋아졌다면, 모델은 **어디에서 task 정보를 얻은 것**인가요? 반대로 결과가 좋아지지 않았더라도 이 실험이 GPT-3 논문의 주장을 반박한다고 할 수 없는 이유를 쓰세요.

**답:**

- **task 정보의 출처:** 모델은 weight update 없이 **prompt 안의 demonstration(context)** 에서 task 정보를 얻습니다. `English: ... / French: ...` 쌍이 반복되는 형식을 보고 "영어 단어 다음에 대응되는 프랑스어를 써야 한다"는 패턴을 forward pass 중에 파악하는 것입니다. 물론 그 패턴을 활용할 수 있는 능력 자체는 pretraining 과정에서 얻은 것이고, 이것이 논문에서 말하는 in-context learning(meta-learning의 inner loop)입니다.
  - 실제 실행 결과, 0-shot에서는 `cheese`를 그대로 반복했고, 1-shot에서는 예시의 답 `loutre de mer`를 그대로 복사했으며, 3-shot에서는 `coterie`처럼 프랑스어처럼 보이는 단어를 만들어 냈습니다(정답 `fromage`는 아님). 예시가 늘수록 출력 형식은 맞춰 가지만 실제 번역 능력은 거의 없다는 것을 알 수 있습니다.
- **논문을 반박할 수 없는 이유:**
  1. **모델 크기 차이:** `distilgpt2`는 약 82M parameter로 GPT-3 175B보다 2000배 이상 작습니다. 논문의 핵심 주장은 in-context learning 능력이 **모델 크기가 커질수록 크게 좋아진다**(scaling에 따른 출현)는 것이고, 실제로 논문에서도 작은 모델들은 few-shot에서 얻는 이득이 매우 작았습니다. 따라서 작은 모델에서 효과가 없는 것은 오히려 논문 결과와 맞습니다.
  2. **학습 데이터와 실험 규모:** distilgpt2는 GPT-3보다 훨씬 적은(그리고 대부분 영어인) 데이터로 학습되어 프랑스어 지식이 부족합니다. 또 query 하나, 예시 3개, greedy decoding만 쓴 단일 샘플 실험이라 통계적으로 의미 있는 평가라고 보기도 어렵습니다.

---
## Part 3. 논문의 주요 결과 읽기

아래는 논문에서 보고한 GPT-3 175B의 일부 결과입니다. 숫자를 직접 계산해 보면서 결과를 해석합니다.

In [6]:
results = {
    "LAMBADA": {"zero": 76.2, "one": 72.5, "few": 86.4, "previous_sota": 68.0},
    "TriviaQA": {"zero": 64.3, "one": 68.0, "few": 71.2, "previous_sota": 68.0},
    "CoQA": {"zero": 81.5, "one": 84.0, "few": 85.0, "previous_sota": 90.7},
}

for task, score in results.items():
    gain = score["few"] - score["zero"]                            # TODO ③ few-shot - zero-shot
    print(f"{task:10s} | zero→few gain: {gain:+.1f} points")

LAMBADA    | zero→few gain: +10.2 points
TriviaQA   | zero→few gain: +6.9 points
CoQA       | zero→few gain: +3.5 points


In [7]:
# ✅ 확인 셀
assert round(results["LAMBADA"]["few"] - results["LAMBADA"]["zero"], 1) == 10.2
assert round(results["TriviaQA"]["few"] - results["TriviaQA"]["zero"], 1) == 6.9
print("✅ 계산 확인")

✅ 계산 확인


### 문제 3-1 (서술형)
1. 세 task 모두에서 `few-shot > zero-shot`인가요? 예외가 있다면 무엇인가요?
2. LAMBADA와 TriviaQA의 결과만 보고 “GPT-3가 모든 NLP task에서 fine-tuned SOTA를 능가했다”고 결론 내려도 될까요? 이유를 쓰세요.

**답:**

1. **few-shot과 zero-shot 비교:** 세 task 모두 few-shot이 zero-shot보다 높습니다(LAMBADA +10.2, TriviaQA +6.9, CoQA +3.5). 다만 **예외는 LAMBADA의 one-shot**입니다. one-shot(72.5)이 zero-shot(76.2)보다 오히려 **3.7점 낮습니다.** 논문은 그 이유를 예시 1개만으로는 "빈칸에 정확히 한 단어를 채우는" 형식(fill-in-the-blank)을 모델이 제대로 알아채지 못하기 때문이라고 설명합니다. 따라서 예시를 추가하면 항상 성능이 오른다고 말할 수는 없습니다. 또 gain의 크기도 task마다 다릅니다(CoQA는 +3.5로 작음).
2. **결론을 내려도 될까?** 안 됩니다.
   - 같은 표 안에서도 **CoQA few-shot(85.0)은 fine-tuned SOTA(90.7)보다 5.7점 낮습니다.**
   - 논문 전체를 보면 ANLI/RTE 같은 NLI, QuAC·DROP·RACE 같은 일부 reading comprehension, WiC 등에서는 GPT-3가 fine-tuned 모델이나 BERT 계열에 크게 뒤처졌습니다.
   - LAMBADA와 TriviaQA는 GPT-3가 특히 잘한 task를 고른 것이라 **선택 편향**이 있습니다. TriviaQA의 비교 대상(68.0)은 retrieval을 쓰는 open-domain 시스템이라 조건도 완전히 같지 않습니다.
   - 따라서 올바른 결론은 "fine-tuning 없이도 **일부 task에서는** fine-tuned SOTA와 비슷하거나 더 높은 성능을 낼 수 있다"까지입니다.

---
## Part 4. Benchmark Contamination 간단 확인

GPT-3는 Common Crawl을 포함한 대규모 웹 데이터로 사전학습되었습니다. 따라서 benchmark의 test 문장이 웹에 공개되어 있었다면 training corpus에 포함될 수 있습니다.

여기서는 실제 GPT-3 training corpus 대신 작은 문자열을 사용해 **n-gram overlap**의 아이디어만 확인합니다.

### 문제 4-1. 두 텍스트 사이의 n-gram overlap을 계산하세요

In [8]:
def word_ngrams(text, n=4):
    words = re.findall(r"[a-z]+", text.lower())
    return {tuple(words[i:i+n]) for i in range(len(words) - n + 1)}

def overlap_ratio(train_text, test_text, n=4):
    train_ng = word_ngrams(train_text, n)
    test_ng = word_ngrams(test_text, n)
    overlap = train_ng & test_ng                         # TODO ④ 두 n-gram 집합의 교집합
    return len(overlap) / max(len(test_ng), 1)

train_text = "the quick brown fox jumps over the lazy dog and runs away"
test_clean = "language models can learn patterns from examples in context"
test_dirty = "the quick brown fox jumps over the fence"

print("clean overlap:", overlap_ratio(train_text, test_clean))
print("dirty overlap:", overlap_ratio(train_text, test_dirty))

clean overlap: 0.0
dirty overlap: 0.8


In [9]:
# ✅ 확인 셀
assert overlap_ratio(train_text, test_clean) == 0
assert overlap_ratio(train_text, test_dirty) > 0
print("✅ overlap 확인")

✅ overlap 확인


### 문제 4-2 (서술형)
Benchmark test example이 pretraining corpus에 포함되어 있다면 왜 성능을 해석하기 어려워지나요? **generalization**과 **memorization**이라는 단어를 사용해 설명하세요.

**답:**

Benchmark는 모델이 **학습 때 보지 못한 데이터에 얼마나 잘 일반화하는지(generalization)** 를 재기 위한 도구입니다. 그런데 test example(질문과 정답, 또는 LAMBADA처럼 지문 전체)이 pretraining corpus에 들어 있으면, 모델이 높은 점수를 받은 이유가 task를 실제로 이해하고 일반화했기 때문인지, 아니면 학습 중에 본 문장과 정답을 **외워서(memorization)** 다시 내놓은 것인지 구분할 수 없게 됩니다. 이 경우 benchmark 점수는 generalization 능력을 **과대평가**하게 되고, 측정하려던 것이 측정되지 않습니다.

GPT-3는 Common Crawl처럼 거대한 웹 데이터로 학습했기 때문에 이 문제가 특히 심각합니다. 논문은 13-gram overlap으로 오염된 example을 찾아 이를 제외한 **clean subset**에서의 성능과 전체 성능을 비교했습니다. 대부분의 task에서는 차이가 작았지만, 일부(예: PIQA, Winograd)에는 표시를 해 두었고, 오염이 심한 일부 데이터셋은 결과 보고에서 제외했습니다. 또 학습 데이터 필터링 과정의 버그 때문에 overlap을 완전히 제거하지 못했다는 점도 한계로 밝혔습니다. 위 실습의 `dirty overlap: 0.8`처럼 test 문장의 4-gram 80%가 train에 있다면, 그 문장에 대한 정답은 generalization의 증거로 보기 어렵습니다.

---
## Part 5. 서술형 복습

**5-1.** GPT-3 논문에서 말하는 *in-context learning*과 일반적인 fine-tuning의 가장 중요한 차이를 설명하세요.

**답:** 가장 중요한 차이는 **task에 적응하는 과정에서 model parameter를 update하느냐**입니다. Fine-tuning은 task별 labeled dataset(보통 수천~수십만 개)으로 gradient update를 해서 weight 자체를 바꾸므로, task마다 별도의 모델이 생기고 큰 데이터셋이 필요하며, 좁은 분포에 overfit되어 spurious correlation을 학습할 위험이 있습니다. 반면 in-context learning은 weight를 고정한 채 **task 설명과 몇 개의 예시를 prompt(context)로만 주고**, 모델이 forward pass 안에서 task를 파악해 답을 생성합니다. 하나의 모델로 여러 task를 처리할 수 있고 예시도 몇 개면 충분하지만, context window 길이(2048 tokens) 때문에 넣을 수 있는 예시 수가 제한되고, 보통 fine-tuned SOTA보다 성능이 낮습니다. 논문은 이를 pretraining(outer loop)에서 얻은 능력을 inference 때(inner loop) 활용하는 meta-learning으로 설명합니다.

**5-2.** 논문에서 125M부터 175B까지 여러 크기의 모델을 함께 학습한 이유는 무엇인가요? 단순히 “가장 큰 모델의 성능을 보기 위해서”보다 구체적으로 쓰세요.

**답:** 모델 크기와 성능 사이의 **scaling 관계를 체계적으로 검증**하기 위해서입니다. (1) Kaplan et al.(2020)이 제시한 "parameter 수가 늘면 validation loss가 power-law를 따라 부드럽게 감소한다"는 경향이 175B까지 이어지는지, 그리고 그 loss 감소가 downstream task 성능 향상으로 이어지는지 확인합니다. (2) 핵심 주장인 **in-context learning 능력이 모델 크기와 함께 커진다**는 점을 보이기 위해서입니다. 크기별로 zero/one/few-shot 성능 차이를 비교하면, 큰 모델일수록 context 속 예시를 더 효과적으로 활용한다는 것(few-shot과 zero-shot 사이 격차가 커짐)을 보여줄 수 있습니다. (3) 산술 연산이나 단어 섞기(word unscrambling)처럼 **일정 크기 이상에서만 갑자기 잘하게 되는 능력**을 관찰할 수 있습니다. (4) Contamination 분석이나 사람이 생성 기사를 판별하는 정확도가 크기에 따라 어떻게 변하는지 보는 등, 결과가 단순히 우연이 아니라 체계적인 경향임을 뒷받침하는 근거가 됩니다. 8개 모델은 같은 데이터와 같은 학습 방식을 사용해서 크기라는 변수만 비교할 수 있도록 설계되었습니다.

**5-3.** GPT-3는 일부 task에서는 강한 few-shot 성능을 보였지만 NLI, 일부 reading comprehension 등에서는 약했습니다. 이 결과가 논문의 주장에 어떤 제한을 주는지 설명하세요.

**답:** "충분히 큰 language model은 fine-tuning 없이 few-shot만으로 범용 task solver가 될 수 있다"는 주장이 **모든 task에 일반화되지는 않는다**는 한계를 보여줍니다. ANLI, RTE 같은 NLI에서는 few-shot 성능이 거의 chance 수준이었고, 두 문장을 비교해야 하는 WiC나 QuAC·DROP·RACE 같은 reading comprehension에서도 fine-tuned 모델보다 크게 낮았습니다. 논문은 그 원인으로 (1) **left-to-right autoregressive 구조**라서 bidirectional 모델처럼 두 문장을 앞뒤로 보면서 비교하거나 긴 지문을 다시 읽어 보는 task에 불리하다는 점, (2) 모든 token을 똑같이 중요하게 다루는 단순한 next-token prediction 목표의 한계, (3) 사람보다 훨씬 많은 텍스트를 봐야 하는 낮은 pretraining sample efficiency를 듭니다. 또 few-shot 성능이 정말 새로운 task를 그 자리에서 "학습"한 결과인지, pretraining 때 본 task를 "인식"하는 것에 가까운지도 분명하지 않습니다. 따라서 GPT-3의 결과는 "scaling이 in-context learning을 크게 개선한다"는 근거는 되지만, fine-tuning을 완전히 대체한다는 결론은 아직 이르며, 구조와 학습 목표 측면의 보완이 필요합니다.

**5-4.** GPT-3가 생성한 뉴스 기사를 사람이 약 52% 정확도로 판별했다는 결과는 어떤 가능성과 위험을 동시에 보여주나요?

**답:** 52%는 무작위로 찍은 수준(50%)과 거의 같으므로, 사람이 GPT-3 175B가 쓴 약 200단어 분량의 기사를 실제 기사와 거의 구별하지 못한다는 뜻입니다. 모델이 작을수록 판별 정확도가 높았고(control 모델 약 86%) 모델이 커질수록 낮아졌습니다.
- **가능성:** 글쓰기 보조, 요약, 초안 작성, 코드·문서 생성 등 사람 수준의 자연스러운 텍스트가 필요한 곳에 폭넓게 쓰일 수 있고, 사람이 하던 많은 언어 작업을 자동화하거나 생산성을 높일 수 있습니다.
- **위험:** 같은 능력이 그대로 **오용**될 수 있습니다. 가짜 뉴스와 대규모 허위정보, spam·phishing, 사회공학적 사기, 학술 부정행위, 여론 조작용 대량 콘텐츠 생성 등의 비용이 크게 낮아집니다. 사람이 판별하지 못하므로 탐지 도구, 출처 표시(provenance) 같은 별도 대응이 필요합니다. 또 학습 데이터에 있는 gender·race·religion 관련 bias가 그럴듯한 문장 속에 섞여 퍼질 수 있고, 사실과 다른 내용을 자신 있게 쓰더라도 독자가 알아채기 어렵습니다. 논문의 Broader Impacts(§6)가 이러한 오용 가능성, 공정성, 에너지 사용 문제를 함께 다룬 이유입니다.

---
## 참고 자료
- Brown et al. (2020), *Language Models are Few-Shot Learners*: Approach (§2), Results (§3), Benchmark contamination (§4), Limitations (§5), Broader Impacts (§6)
- OpenAI GPT-3 repository: 논문의 synthetic task data와 benchmark overlap 관련 자료가 공개되어 있습니다.

> 이 과제는 GPT-3 175B의 원 실험을 재현하는 과제가 아니라, 논문의 핵심 메커니즘과 결과 해석을 작은 예제로 복습하기 위한 과제입니다.